In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Smart MCQ Solver — Model 2**
## BiLSTM + Attention (Built From Scratch)

## Our second main model — built entirely **from scratch**, with no pretrained weights anywhere. Word embeddings, the BiLSTM encoder, the attention mechanism, and the scoring head are all trained from random initialization on this competition's data only.



## Imports & Configuration

In [21]:
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings("ignore")

import wandb

In [22]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABELS = ["A", "B", "C", "D", "E"]
print("Using device:", DEVICE)

Using device: cuda


## Load Dataset 

In [23]:
TRAIN_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUB_CSV = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

train_df_full = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

print("Train shape:", train_df_full.shape)
print("Test shape:", test_df.shape)
train_df_full.head()

Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## **Tokenizer & Vocabulary (From Scratch)**
## No external tokenizer is used — a simple regex tokenizer and a vocabulary built directly from the training data. 

In [24]:
TOKEN_RE = re.compile(r"[A-Za-z]+|[0-9]+|[^\sA-Za-z0-9]")

def tokenize(text: str):
    text = str(text).lower()
    return TOKEN_RE.findall(text)

In [25]:
class Vocab:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.stoi = {"<pad>": 0, "<unk>": 1}
        self.itos = ["<pad>", "<unk>"]

    def build(self, texts):
        from collections import Counter
        counter = Counter()
        for t in texts:
            counter.update(tokenize(t))
        for word, freq in counter.items():
            if freq >= self.min_freq and word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)
        return self

    def encode(self, text, max_len):
        ids = [self.stoi.get(tok, 1) for tok in tokenize(text)][:max_len]
        if len(ids) < max_len:
            ids = ids + [0] * (max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.itos)

## Build Vocab 

In [26]:
train_df, val_df = train_test_split(train_df_full, test_size=0.15, random_state=SEED)

text_cols = ["prompt", "A", "B", "C", "D", "E"]
all_texts = pd.concat([train_df[c] for c in text_cols]).tolist()

vocab = Vocab(min_freq=1).build(all_texts)
print("Vocabulary size:", len(vocab))

Vocabulary size: 3005


## Dataset Class 

In [27]:
MAX_PROMPT_LEN = 64
MAX_OPT_LEN = 16


class MCQDataset(Dataset):
    def __init__(self, df, vocab, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = torch.tensor(self.vocab.encode(row["prompt"], MAX_PROMPT_LEN))
        opt_ids = torch.stack([
            torch.tensor(self.vocab.encode(row[l], MAX_OPT_LEN)) for l in LABELS
        ])

        item = {"prompt": prompt_ids, "options": opt_ids}
        if self.has_labels:
            item["label"] = torch.tensor(LABELS.index(row["answer"]))
        else:
            item["id"] = row["id"]
        return item

## DataLoaders 

In [28]:
BATCH_SIZE = 32
train_ds = MCQDataset(train_df, vocab)
val_ds = MCQDataset(val_df, vocab)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)


## **Model Architecture**
##  Design: shared BiLSTM encodes both the prompt and each option; an attention layer lets each option attend over the most relevant part of the prompt; match features (concat, |diff|, product) feed an MLP scorer that ranks all 5 options.

In [29]:
class ScratchMCQModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.encoder = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn_W = nn.Linear(hidden_dim * 2, hidden_dim * 2, bias=False)
        match_dim = hidden_dim * 2 * 4
        self.scorer = nn.Sequential(
            nn.Linear(match_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
    def encode(self, x):
        emb = self.embedding(x)
        out, _ = self.encoder(emb)
        return out
    def masked_mean(self, seq, ids):
        mask = (ids != 0).unsqueeze(-1).float()
        summed = (seq * mask).sum(1)
        count = mask.sum(1).clamp(min=1)
        return summed / count
    def forward(self, prompt_ids, option_ids):
        B, num_opts, Lo = option_ids.shape
        prompt_enc = self.encode(prompt_ids)
        prompt_vec = self.masked_mean(prompt_enc, prompt_ids)
        scores = []
        for i in range(num_opts):
            opt_ids_i = option_ids[:, i, :]
            opt_enc = self.encode(opt_ids_i)
            opt_vec = self.masked_mean(opt_enc, opt_ids_i)
            attn_scores = torch.bmm(self.attn_W(prompt_enc), opt_vec.unsqueeze(-1)).squeeze(-1)
            pad_mask = (prompt_ids == 0)
            attn_scores = attn_scores.masked_fill(pad_mask, -1e9)
            attn_weights = F.softmax(attn_scores, dim=-1).unsqueeze(-1)
            prompt_ctx = (attn_weights * prompt_enc).sum(1)
            diff = torch.abs(opt_vec - prompt_ctx)
            prod = opt_vec * prompt_ctx
            match_vec = torch.cat([opt_vec, prompt_ctx, diff, prod], dim=-1)
            score_i = self.scorer(match_vec).squeeze(-1)
            scores.append(score_i)
        return torch.stack(scores, dim=1)

## **Initialize Model** 

In [30]:
model = ScratchMCQModel(vocab_size=len(vocab), emb_dim=128, hidden_dim=128, dropout=0.3).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")

Trainable parameters: 845,697


## **Metrics & Train/Eval Functions**  

In [31]:
def map_at_3(logits, labels):
    top3 = torch.topk(logits, k=3, dim=-1).indices
    scores = []
    for pred, true in zip(top3.tolist(), labels.tolist()):
        if true in pred:
            rank = pred.index(true) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return sum(scores) / len(scores)

In [32]:
def accuracy_f1(logits, labels):
    preds = torch.argmax(logits, dim=-1).cpu().numpy()
    labels_np = labels.cpu().numpy()
    return (
        accuracy_score(labels_np, preds),
        f1_score(labels_np, preds, average="macro"),
    )

In [33]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_logits, all_labels = 0.0, [], []
    for batch in loader:
        prompt = batch["prompt"].to(DEVICE)
        options = batch["options"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        with torch.set_grad_enabled(is_train):
            logits = model(prompt, options)
            loss = F.cross_entropy(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        total_loss += loss.item() * prompt.size(0)
        all_logits.append(logits.detach().cpu())
        all_labels.append(labels.detach().cpu())

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    acc, f1 = accuracy_f1(all_logits, all_labels)
    map3 = map_at_3(all_logits, all_labels)
    return total_loss / len(loader.dataset), acc, f1, map3

## **W&B Login** 

In [34]:
from kaggle_secrets import UserSecretsClient
import wandb
import os

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb_api_key")
os.environ["WANDB_API_KEY"] = wandb_key
wandb.login()

True

## **Train** 

In [35]:
EPOCHS = 15
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

wandb.init(
    project="smart-mcq-solver",
    name="model-from-scratch-bilstm-attn",
    config={
        "emb_dim": 128, "hidden_dim": 128, "lr": LR,
        "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "model_type": "from_scratch_bilstm_attention",
    },
)

best_map3 = 0.0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1, tr_map3 = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1, val_map3 = run_epoch(model, val_loader)

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} "
          f"f1={tr_f1:.4f} map3={tr_map3:.4f} | val_loss={val_loss:.4f} "
          f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} val_map3={val_map3:.4f}")

    wandb.log({
        "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
        "train_f1": tr_f1, "train_map3": tr_map3,
        "val_loss": val_loss, "val_acc": val_acc,
        "val_f1": val_f1, "val_map3": val_map3,
    })

    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save({"model_state": model.state_dict(), "vocab_stoi": vocab.stoi},
                   "scratch_model_best.pt")
        print(f"  -> New best model saved (val_map3={val_map3:.4f})")

wandb.finish()
print("Best validation MAP@3:", best_map3)

Epoch 01 | train_loss=1.2478 acc=0.4994 f1=0.4938 map3=0.6424 | val_loss=0.5694 val_acc=0.7733 val_f1=0.7781 val_map3=0.8356
  -> New best model saved (val_map3=0.8356)
Epoch 02 | train_loss=0.4119 acc=0.8082 f1=0.8050 map3=0.8791 | val_loss=0.2515 val_acc=0.8800 val_f1=0.8873 val_map3=0.9000
  -> New best model saved (val_map3=0.9000)
Epoch 03 | train_loss=0.2879 acc=0.8400 f1=0.8382 map3=0.8987 | val_loss=0.2404 val_acc=0.8767 val_f1=0.8843 val_map3=0.8978
Epoch 04 | train_loss=0.2696 acc=0.8424 f1=0.8406 map3=0.9036 | val_loss=0.2315 val_acc=0.8767 val_f1=0.8833 val_map3=0.8967
Epoch 05 | train_loss=0.2666 acc=0.8629 f1=0.8605 map3=0.9126 | val_loss=0.2312 val_acc=0.8867 val_f1=0.8932 val_map3=0.9033
  -> New best model saved (val_map3=0.9033)
Epoch 06 | train_loss=0.2671 acc=0.8447 f1=0.8442 map3=0.9040 | val_loss=0.2331 val_acc=0.8767 val_f1=0.8833 val_map3=0.8967
Epoch 07 | train_loss=0.2669 acc=0.8447 f1=0.8430 map3=0.9038 | val_loss=0.2292 val_acc=0.8800 val_f1=0.8869 val_map3=

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▇█████████████
train_f1,▁▇█████████████
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train_map3,▁▇█████████████
val_acc,▁█▇▇█▇█▇███████
val_f1,▁█▇▇█▇█▇███████
val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map3,▁█▇▇█▇█▇███████
epoch,15
train_acc,0.85294


Best validation MAP@3: 0.9033333333333333


## **Load Best Checkpoint** 

In [36]:
checkpoint = torch.load("scratch_model_best.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])

val_loss, val_acc, val_f1, val_map3 = run_epoch(model, val_loader)
print(f"Best model -> val_acc={val_acc:.4f}, val_f1={val_f1:.4f}, val_map3={val_map3:.4f}")

Best model -> val_acc=0.8867, val_f1=0.8932, val_map3=0.9033


## **Inference** 

In [37]:
class MCQTestDataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = torch.tensor(self.vocab.encode(row["prompt"], MAX_PROMPT_LEN))
        opt_ids = torch.stack([
            torch.tensor(self.vocab.encode(row[l], MAX_OPT_LEN)) for l in LABELS
        ])
        return {"id": row["id"], "prompt": prompt_ids, "options": opt_ids}


test_ds = MCQTestDataset(test_df, vocab)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model.eval()
ids_all, preds_all = [], []
with torch.no_grad():
    for batch in test_loader:
        prompt = batch["prompt"].to(DEVICE)
        options = batch["options"].to(DEVICE)
        logits = model(prompt, options)
        top3 = torch.topk(logits, k=3, dim=-1).indices.cpu().numpy()
        preds_all.extend([" ".join(LABELS[i] for i in row) for row in top3])
        ids_all.extend(batch["id"].numpy())

print("Sample predictions:", preds_all[:5])

Sample predictions: ['A C B', 'B E D', 'B A E', 'E C A', 'C B A']


## **Submission**

In [38]:
submission = pd.DataFrame({"ID": ids_all, "Prediction": preds_all})
submission.to_csv("submission.csv", index=False)

print("Submission saved: submission.csv")
submission.head()

Submission saved: submission.csv


,ID,Prediction
0,1,A C B
1,2,B E D
2,3,B A E
3,4,E C A
4,5,C B A


---
## **Summary**

| Metric | Value |
|---|---|
| Model | BiLSTM + Attention (From Scratch) |
| Category | Model built from scratch (no pretrained weights) |
| Training required | Yes (fully from random initialization) |
| Kaggle MAP@3 | 0.71238 (initial run; retrain on full data aims higher) |

## No pretrained weights, embeddings, or tokenizers are used anywhere in this model — the vocabulary, embeddings, encoder, attention, and scoring head are all learned entirely from this competition's training data.
